In [1]:
import json, time, gc
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from lifelines.utils import concordance_index
import warnings
warnings.filterwarnings("ignore")

RAW_ROOT   = Path("data/raw")
PROC_DIR   = Path("data/processed")
BASE_DIR   = Path("data/models/baseline_pinned")
FUZZY_DIR  = Path("data/fuzzy")
AUG_DIR    = Path("data/models/augmented")
AUG_DIR.mkdir(parents=True, exist_ok=True)

with open(PROC_DIR / "task_registry.json") as f:
    TASKS = json.load(f)
with open(PROC_DIR / "feature_list.json") as f:
    FEAT = json.load(f)

FEATURES    = FEAT["features"]
CATEGORICAL = [c for c in ["userCoinTypeMode", "userReserveMode"] if c in FEATURES]
FUZZY_COLS  = ["mu_Safe", "mu_Watch", "mu_Danger", "mu_Critical", "interval_width"]

HOLDOUT_START = pd.Timestamp("2024-04-01").timestamp()
HOLDOUT_END   = pd.Timestamp("2024-07-02").timestamp()

PARAMS = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "verbosity": 0,
}

In [2]:
def xgb_cox_label(y_time, y_status):
    y = y_time.astype(float).copy()
    y[y_status == 0] = -y[y_status == 0]
    y[y == 0] = 1e-6
    return y

def load_task_with_fuzzy(task):
    tid = task["task_id"]
    cols = FEATURES + CATEGORICAL + ["timeDiff", "status", "timestamp"]
    cols = list(dict.fromkeys(cols))
    df = pd.read_csv(RAW_ROOT / task["train_path"], usecols=cols)
    for c in CATEGORICAL:
        le = LabelEncoder().fit(df[c].astype(str))
        df[c] = le.transform(df[c].astype(str))
    df[FEATURES] = df[FEATURES].fillna(0)

    fuzzy = pd.read_parquet(FUZZY_DIR / f"{tid}_memberships.parquet")
    # Alignment check: row count must match (fuzzy was computed on the exact train.csv order)
    assert len(fuzzy) == len(df), f"length mismatch {len(fuzzy)} vs {len(df)}"

    for col in FUZZY_COLS:
        df[col] = fuzzy[col].values

    fit     = df[df["timestamp"] <  HOLDOUT_START].reset_index(drop=True)
    holdout = df[(df["timestamp"] >= HOLDOUT_START) & (df["timestamp"] <= HOLDOUT_END)].reset_index(drop=True)
    return fit, holdout

In [3]:
def fit_and_eval(fit, holdout, features):
    fit = fit.sort_values("timestamp").reset_index(drop=True)
    n = len(fit)
    n_val = int(n * 0.1)
    tr = fit.iloc[:n - n_val]
    vl = fit.iloc[n - n_val:]

    y_tr = xgb_cox_label(tr["timeDiff"].values, tr["status"].values)
    y_vl = xgb_cox_label(vl["timeDiff"].values, vl["status"].values)

    dtr = xgb.DMatrix(tr[features].values, label=y_tr)
    dvl = xgb.DMatrix(vl[features].values, label=y_vl)
    dho = xgb.DMatrix(holdout[features].values)

    bst = xgb.train(PARAMS, dtr, num_boost_round=300,
                    evals=[(dvl, "val")], early_stopping_rounds=20, verbose_eval=False)

    risk_ho = bst.predict(dho)
    c_ho = concordance_index(holdout["timeDiff"].values, -risk_ho, holdout["status"].values)
    return bst, risk_ho, c_ho, bst.best_iteration

AUG_FEATURES = FEATURES + FUZZY_COLS

results = []
t_start = time.time()
for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    print(f"[{i:2d}/16] {tid}  loading", end=" ")
    t0 = time.time()
    fit, holdout = load_task_with_fuzzy(task)
    print(f"({len(fit):,} fit, {len(holdout):,} holdout, {time.time()-t0:.0f}s)")

    print(f"       fitting augmented ({len(AUG_FEATURES)} features)", end=" ")
    t0 = time.time()
    bst, risk_ho, c_ho, best_iter = fit_and_eval(fit, holdout, AUG_FEATURES)
    dt = time.time() - t0
    print(f"({best_iter} rounds, hold-out C={c_ho:.4f}, {dt:.0f}s)")

    bst.save_model(str(AUG_DIR / f"{tid}.ubj"))
    pd.DataFrame({
        "risk": risk_ho,
        "timeDiff": holdout["timeDiff"].values,
        "status":   holdout["status"].values,
        "timestamp": holdout["timestamp"].values,
    }).to_csv(AUG_DIR / f"{tid}_holdout_preds.csv", index=False)

    results.append({
        "task_id": tid,
        "n_holdout": len(holdout),
        "holdout_c_index_aug": float(c_ho),
        "best_iter": int(best_iter),
        "fit_seconds": dt,
    })
    del fit, holdout, bst; gc.collect()

print(f"\ntotal time: {(time.time()-t_start)/60:.1f} min")

[ 1/16] Borrow_Deposit  loading (784,167 fit, 98,041 holdout, 19s)
       fitting augmented (85 features) (34 rounds, hold-out C=0.7130, 20s)
[ 2/16] Borrow_Repay  loading (757,265 fit, 88,543 holdout, 19s)
       fitting augmented (85 features) (152 rounds, hold-out C=0.8234, 34s)
[ 3/16] Borrow_Withdraw  loading (785,318 fit, 98,106 holdout, 20s)
       fitting augmented (85 features) (94 rounds, hold-out C=0.7314, 26s)
[ 4/16] Borrow_Liquidated  loading (786,960 fit, 98,420 holdout, 20s)
       fitting augmented (85 features) (47 rounds, hold-out C=0.8126, 18s)
[ 5/16] Deposit_Borrow  loading (1,725,041 fit, 440,300 holdout, 50s)
       fitting augmented (85 features) (75 rounds, hold-out C=0.9173, 64s)
[ 6/16] Deposit_Repay  loading (1,722,401 fit, 440,439 holdout, 49s)
       fitting augmented (85 features) (299 rounds, hold-out C=0.9547, 113s)
[ 7/16] Deposit_Withdraw  loading (1,714,498 fit, 253,694 holdout, 46s)
       fitting augmented (85 features) (201 rounds, hold-out C=0.8

In [4]:
base_df = pd.read_csv(BASE_DIR / "pinned_baseline_summary.csv")[["task_id", "holdout_c_index"]].rename(
    columns={"holdout_c_index": "holdout_c_index_base"}
)
aug_df = pd.DataFrame(results)[["task_id", "holdout_c_index_aug"]]
cmp = base_df.merge(aug_df, on="task_id")
cmp["delta"] = cmp["holdout_c_index_aug"] - cmp["holdout_c_index_base"]

print(cmp.round(4).to_string(index=False))
mean_base = cmp["holdout_c_index_base"].mean()
mean_aug  = cmp["holdout_c_index_aug"].mean()
delta     = mean_aug - mean_base
print(f"\nmean baseline C-index:  {mean_base:.4f}")
print(f"mean augmented C-index: {mean_aug:.4f}")
print(f"mean delta:             {delta:+.4f}")
cmp.to_csv(AUG_DIR / "augmented_vs_baseline.csv", index=False)

            task_id  holdout_c_index_base  holdout_c_index_aug   delta
     Borrow_Deposit                0.7115               0.7130  0.0015
       Borrow_Repay                0.8254               0.8234 -0.0020
    Borrow_Withdraw                0.7115               0.7314  0.0199
  Borrow_Liquidated                0.8043               0.8126  0.0083
     Deposit_Borrow                0.9260               0.9173 -0.0087
      Deposit_Repay                0.9548               0.9547 -0.0001
   Deposit_Withdraw                0.8984               0.8975 -0.0009
 Deposit_Liquidated                0.9406               0.9446  0.0040
       Repay_Borrow                0.8087               0.8073 -0.0014
      Repay_Deposit                0.7140               0.7093 -0.0047
     Repay_Withdraw                0.6876               0.6967  0.0091
   Repay_Liquidated                0.7997               0.8227  0.0230
    Withdraw_Borrow                0.9806               0.9780 -0.0026
   Wit

In [8]:
import numpy as np
import pandas as pd
from lifelines.utils import concordance_index

N_BOOT = 100
BLOCK_SIZE = 10
MAX_BOOT_N = 3000  # cap sample size per bootstrap draw

def block_bootstrap_delta(preds_base, preds_aug, n_boot=N_BOOT,
                          block_size=BLOCK_SIZE, max_n=MAX_BOOT_N, seed=42):
    rng = np.random.default_rng(seed)
    y  = preds_base["timeDiff"].values
    e  = preds_base["status"].values
    rb = preds_base["risk"].values
    ra = preds_aug["risk"].values
    n_full = len(y)
    n = min(n_full, max_n)
    deltas = []
    for _ in range(n_boot):
        idx = []
        while len(idx) < n:
            start = rng.integers(0, n_full)
            length = rng.geometric(1.0 / block_size)
            for k in range(length):
                idx.append((start + k) % n_full)
                if len(idx) >= n:
                    break
        idx = np.array(idx[:n])
        if e[idx].sum() < 5:
            continue
        try:
            cb = concordance_index(y[idx], -rb[idx], e[idx])
            ca = concordance_index(y[idx], -ra[idx], e[idx])
            deltas.append(ca - cb)
        except Exception:
            continue
    return np.array(deltas)

boot_rows = []
for i, tid in enumerate(TASKS, 1):
    base_path = BASE_DIR / f"{tid}_holdout_preds.csv"
    aug_path  = AUG_DIR  / f"{tid}_holdout_preds.csv"
    if not (base_path.exists() and aug_path.exists()):
        continue
    pb = pd.read_csv(base_path)
    pa = pd.read_csv(aug_path)
    print(f"[{i}/{len(TASKS)}] {tid}  n_holdout={len(pb)}  bootstrapping...", flush=True)
    d = block_bootstrap_delta(pb, pa)
    if len(d) == 0:
        continue
    mean_d = float(d.mean())
    lo, hi = np.quantile(d, [0.025, 0.975])
    p_pos  = float((d > 0).mean())
    if lo > 0:
        verdict = "WIN"
    elif hi < 0:
        verdict = "LOSE"
    else:
        verdict = "null"
    print(f"       delta={mean_d:+.4f}  CI=[{lo:+.4f},{hi:+.4f}]  p_pos={p_pos:.2f}  {verdict}", flush=True)
    boot_rows.append({
        "task_id": tid,
        "delta_mean": mean_d,
        "ci_lo": lo, "ci_hi": hi,
        "p_pos": p_pos,
        "verdict": verdict,
    })

boot_df = pd.DataFrame(boot_rows)
print()
print(boot_df.round(4).to_string(index=False))

mean_delta_across_tasks = boot_df["delta_mean"].mean()
n_wins   = (boot_df["verdict"] == "WIN").sum()
n_losses = (boot_df["verdict"] == "LOSE").sum()
n_null   = (boot_df["verdict"] == "null").sum()

print(f"\nacross 16 tasks:")
print(f"  mean delta:  {mean_delta_across_tasks:+.4f}")
print(f"  wins:        {n_wins}")
print(f"  losses:      {n_losses}")
print(f"  null:        {n_null}")

print(f"\npre-registered decision:")
if n_wins > n_losses and mean_delta_across_tasks > 0:
    print("  POSITIVE: fuzzy features add predictive value on majority of tasks")
elif n_losses > n_wins:
    print("  NEGATIVE: fuzzy features significantly hurt on majority of tasks")
else:
    print("  NULL: no clear direction, fuzzy features do not change performance")

boot_df.to_csv(AUG_DIR / "paired_bootstrap.csv", index=False)


Empty DataFrame
Columns: []
Index: []


KeyError: 'delta_mean'

In [10]:
import numpy as np
import pandas as pd
from lifelines.utils import concordance_index

N_BOOT = 100
BLOCK_SIZE = 10
MAX_BOOT_N = 3000

first_tids = [t["task_id"] for t in TASKS[:3]]
print(f"BASE_DIR: {BASE_DIR}")
print(f"AUG_DIR:  {AUG_DIR}")
print(f"TASKS count: {len(TASKS)}")
print(f"first 3 task ids: {first_tids}")
print()

def block_bootstrap_delta(preds_base, preds_aug, n_boot=N_BOOT,
                          block_size=BLOCK_SIZE, max_n=MAX_BOOT_N, seed=42):
    rng = np.random.default_rng(seed)
    y  = preds_base["timeDiff"].values
    e  = preds_base["status"].values
    rb = preds_base["risk"].values
    ra = preds_aug["risk"].values
    n_full = len(y)
    n = min(n_full, max_n)
    deltas = []
    for _ in range(n_boot):
        idx = []
        while len(idx) < n:
            start = rng.integers(0, n_full)
            length = rng.geometric(1.0 / block_size)
            for k in range(length):
                idx.append((start + k) % n_full)
                if len(idx) >= n:
                    break
        idx = np.array(idx[:n])
        if e[idx].sum() < 5:
            continue
        try:
            cb = concordance_index(y[idx], -rb[idx], e[idx])
            ca = concordance_index(y[idx], -ra[idx], e[idx])
            deltas.append(ca - cb)
        except Exception:
            continue
    return np.array(deltas)

boot_rows = []
skipped = []
for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    base_path = BASE_DIR / f"{tid}_holdout_preds.csv"
    aug_path  = AUG_DIR  / f"{tid}_holdout_preds.csv"
    if not (base_path.exists() and aug_path.exists()):
        skipped.append(tid)
        continue
    pb = pd.read_csv(base_path)
    pa = pd.read_csv(aug_path)
    print(f"[{i}/{len(TASKS)}] {tid}  n_holdout={len(pb)}  bootstrapping...", flush=True)
    d = block_bootstrap_delta(pb, pa)
    if len(d) == 0:
        print(f"       no valid bootstrap samples")
        continue
    mean_d = float(d.mean())
    lo, hi = np.quantile(d, [0.025, 0.975])
    p_pos  = float((d > 0).mean())
    if lo > 0:
        verdict = "WIN"
    elif hi < 0:
        verdict = "LOSE"
    else:
        verdict = "null"
    print(f"       delta={mean_d:+.4f}  CI=[{lo:+.4f},{hi:+.4f}]  p_pos={p_pos:.2f}  {verdict}", flush=True)
    boot_rows.append({
        "task_id": tid,
        "delta_mean": mean_d,
        "ci_lo": lo, "ci_hi": hi,
        "p_pos": p_pos,
        "verdict": verdict,
    })

if skipped:
    print(f"\nskipped {len(skipped)}: {skipped}")

boot_df = pd.DataFrame(boot_rows)
print()
print(boot_df.round(4).to_string(index=False))

mean_delta_across_tasks = boot_df["delta_mean"].mean()
n_wins   = (boot_df["verdict"] == "WIN").sum()
n_losses = (boot_df["verdict"] == "LOSE").sum()
n_null   = (boot_df["verdict"] == "null").sum()

print(f"\nacross {len(boot_df)} tasks:")
print(f"  mean delta:  {mean_delta_across_tasks:+.4f}")
print(f"  wins:        {n_wins}")
print(f"  losses:      {n_losses}")
print(f"  null:        {n_null}")

print(f"\npre-registered decision:")
if n_wins > n_losses and mean_delta_across_tasks > 0:
    print("  POSITIVE: fuzzy features add predictive value on majority of tasks")
elif n_losses > n_wins:
    print("  NEGATIVE: fuzzy features significantly hurt on majority of tasks")
else:
    print("  NULL: no clear direction, fuzzy features do not change performance")

boot_df.to_csv(AUG_DIR / "paired_bootstrap.csv", index=False)

BASE_DIR: data\models\baseline_pinned
AUG_DIR:  data\models\augmented
TASKS count: 16
first 3 task ids: ['Borrow_Deposit', 'Borrow_Repay', 'Borrow_Withdraw']

[1/16] Borrow_Deposit  n_holdout=98041  bootstrapping...
       delta=+0.0017  CI=[-0.0033,+0.0060]  p_pos=0.74  null
[2/16] Borrow_Repay  n_holdout=88543  bootstrapping...
       delta=-0.0021  CI=[-0.0042,+0.0004]  p_pos=0.03  null
[3/16] Borrow_Withdraw  n_holdout=98106  bootstrapping...
       delta=+0.0195  CI=[+0.0138,+0.0268]  p_pos=1.00  WIN
[4/16] Borrow_Liquidated  n_holdout=98420  bootstrapping...
       delta=+0.0103  CI=[-0.0474,+0.1017]  p_pos=0.56  null
[5/16] Deposit_Borrow  n_holdout=440300  bootstrapping...
       delta=-0.0085  CI=[-0.0146,-0.0033]  p_pos=0.00  LOSE
[6/16] Deposit_Repay  n_holdout=440439  bootstrapping...
       delta=-0.0001  CI=[-0.0017,+0.0022]  p_pos=0.42  null
[7/16] Deposit_Withdraw  n_holdout=253694  bootstrapping...
       delta=-0.0008  CI=[-0.0027,+0.0007]  p_pos=0.21  null
[8/16] Dep